# Computing Eigenvectors and Eigenvalues

This notebook focuses on understanding and implementing the computation of eigenvectors and eigenvalues from first principles.

## Key Concepts
- **Eigenvalues**: Scalar values λ such that Av = λv
- **Eigenvectors**: Non-zero vectors v such that Av = λv where λ is an eigenvalue
- **Characteristic Equation**: det(A - λI) = 0, used to find eigenvalues
- **Characteristic Polynomial**: The polynomial obtained from det(A - λI)

## Imports and Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Compute Characteristic Polynomial

Solves det(A - λI) = 0 to find eigenvalues. 

**Why is the determinant zero?**

Starting from the eigenvalue definition **Av = λv**, we can rearrange to get:
- Av - λv = 0
- (A - λI)v = 0  (factoring out v, where I is the identity matrix)

For a non-trivial solution v ≠ 0 to exist, the matrix (A - λI) must be singular (non-invertible). A matrix is singular if and only if its determinant is zero. Therefore: **det(A - λI) = 0**

The values of λ that satisfy this equation are the eigenvalues of A.

Computes the characteristic polynomial coefficients and finds its roots.
The roots of det(A - λI) = 0 are the eigenvalues of the matrix A.

In [2]:
def compute_characteristic_polynomial(A):
    """
    Compute eigenvalues by solving the characteristic equation det(A - λI) = 0.
    
    The mathematical approach:
    1. Form the matrix (A - λI) where λ is a scalar variable
    2. Compute det(A - λI) to get the characteristic polynomial
    3. Find roots of the characteristic polynomial to get eigenvalues
    
    Parameters:
    -----------
    A : ndarray
        Square matrix of shape (n, n)
    
    Returns:
    --------
    eigenvalues : ndarray
        Array of eigenvalues (roots of det(A - λI) = 0)
    """
    # Step 1: Compute characteristic polynomial coefficients using np.poly
    # np.poly(A) returns coefficients of det(λI - A) in decreasing order
    char_poly_coeffs = np.poly(A)
    
    # Step 2: Find roots of the characteristic polynomial
    # The roots of det(A - λI) = 0 are the eigenvalues
    eigenvalues = np.roots(char_poly_coeffs)
    
    return eigenvalues

## Find Eigenvector for Single Eigenvalue

Finds an eigenvector v for a given eigenvalue λ using the formula **Av = λv**.
Constructs the matrix (A - λI), converts it to a solvable reduced system, and solves it using `np.linalg.solve()`.
The result is normalized to have unit length for comparison purposes.

In [3]:
def find_eigenvector_for_eigenvalue(A, eigenvalue):
    """
    Find an eigenvector for a given eigenvalue.
    
    Solves the eigenvector equation Av = λv, which is equivalent to:
    (A - λI)v = 0
    
    We create a non-trivial solution by setting one component to 1
    and solving for the remaining components using np.linalg.solve.
    
    Parameters:
    -----------
    A : ndarray
        Square matrix of shape (n, n)
    eigenvalue : float or complex
        The eigenvalue for which to find the eigenvector
    
    Returns:
    --------
    eigenvector : ndarray
        An eigenvector v satisfying Av = λv
    """
    n = A.shape[0]
    
    # Step 1: Create (A - λI)
    I = np.eye(n)
    A_minus_lambda_I = A - eigenvalue * I
    
    # Step 2: Solve the homogeneous system (A - λI)v = 0
    # We create a non-trivial solution by setting the last component to 1
    # and solving for the remaining components
    
    # Use the last component as a constraint: set it to 1
    # Rearrange: [A - λI]_reduced * v_reduced = -[A - λI]_last_col
    
    A_reduced = A_minus_lambda_I[:-1, :-1]  # All rows and columns except last
    b = -A_minus_lambda_I[:-1, -1]  # Right-hand side from last column
    
    # Solve the reduced system
    v_reduced = np.linalg.solve(A_reduced, b)
    
    # Construct full eigenvector by appending the constraint component (1)
    eigenvector = np.append(v_reduced, 1.0)
    
    return eigenvector

## Main Function: Compute Eigenvalues and Eigenvectors

In [4]:
def compute_eigensystem(A):
    """
    Compute all eigenvalues and corresponding eigenvectors of matrix A.
    
    Uses the definition: Av = λv
    - Step 1: Find eigenvalues λ by solving det(A - λI) = 0
    - Step 2: For each λ, find eigenvectors v by solving Av = λv (or equivalently (A - λI)v = 0)
    
    Parameters:
    -----------
    A : ndarray
        Square matrix of shape (n, n)
    
    Returns:
    --------
    eigenvalues : ndarray
        Array of eigenvalues
    eigenvectors : ndarray
        Matrix where columns are eigenvectors.
        Column i is the eigenvector corresponding to eigenvalues[i]
    """
    # Step 1: Find all eigenvalues by solving det(A - λI) = 0
    eigenvalues = compute_characteristic_polynomial(A)
    
    # Step 2: For each eigenvalue, find corresponding eigenvector using Av = λv
    n = A.shape[0]
    eigenvectors = np.zeros((n, n), dtype=complex)
    
    for i, eigenvalue in enumerate(eigenvalues):
        eigenvector = find_eigenvector_for_eigenvalue(A, eigenvalue)
        # Normalize the eigenvector to unit length
        eigenvector = eigenvector / np.linalg.norm(eigenvector)
        eigenvectors[:, i] = eigenvector
    
    return eigenvalues, eigenvectors

## Example 1: 2x2 Matrix

## Understanding the Determinant Method

To find eigenvalues, we solve the characteristic equation: **det(A - λI) = 0**

The process:
1. **Create (A - λI)**: Subtract λ times the identity matrix from A
2. **Compute determinant**: Calculate det(A - λI) as a polynomial in λ (characteristic polynomial)
3. **Find roots**: The roots of this polynomial are the eigenvalues

For a 2×2 matrix, the characteristic polynomial is quadratic and we can find roots analytically.
For larger matrices, we use numerical root-finding methods.

In [5]:
# Demonstration of the Determinant Method for a Simple 2x2 Matrix
print("="*60)
print("DETERMINANT METHOD FOR FINDING EIGENVALUES")
print("="*60)

# Simple 2x2 example
A_demo = np.array([
    [4, 1],
    [2, 3]
], dtype=float)

print()
print("Given Matrix A:")
print(A_demo)
print()
print("Find eigenvalues by solving det(A - λI) = 0")

# For a 2x2 matrix [a, b; c, d]:
# A - λI = [a-λ, b; c, d-λ]
# det(A - λI) = (a-λ)(d-λ) - bc
#             = λ² - (a+d)λ + (ad-bc)
#             = λ² - trace(A)λ + det(A)

trace_A = np.trace(A_demo)
det_A = np.linalg.det(A_demo)

print()
print(f"Trace(A) = sum of diagonal elements = {trace_A}")
print(f"det(A) = {det_A:.4f}")

print()
print("Characteristic polynomial: λ² - trace(A)λ + det(A) = 0")
print(f"                         λ² - {trace_A}λ + {det_A:.4f} = 0")

# Find roots using quadratic formula
a_coef = 1
b_coef = -trace_A
c_coef = det_A

discriminant = b_coef**2 - 4*a_coef*c_coef
lambda1 = (-b_coef + np.sqrt(discriminant)) / (2*a_coef)
lambda2 = (-b_coef - np.sqrt(discriminant)) / (2*a_coef)

print()
print(f"Using quadratic formula:")
print(f"λ₁ = {lambda1:.6f}")
print(f"λ₂ = {lambda2:.6f}")

# Verify with numpy
numpy_eigenvalues = np.linalg.eigvals(A_demo)
print()
print(f"Verification with np.linalg.eigvals():")
print(f"Eigenvalues: {numpy_eigenvalues}")
print()
print("✓ Both methods give the same eigenvalues!")


DETERMINANT METHOD FOR FINDING EIGENVALUES

Given Matrix A:
[[4. 1.]
 [2. 3.]]

Find eigenvalues by solving det(A - λI) = 0

Trace(A) = sum of diagonal elements = 7.0
det(A) = 10.0000

Characteristic polynomial: λ² - trace(A)λ + det(A) = 0
                         λ² - 7.0λ + 10.0000 = 0

Using quadratic formula:
λ₁ = 5.000000
λ₂ = 2.000000

Verification with np.linalg.eigvals():
Eigenvalues: [5. 2.]

✓ Both methods give the same eigenvalues!


In [6]:
# Define a simple 2x2 matrix
A_2x2 = np.array([
    [4, -2],
    [1, 1]
], dtype=float)

print("Matrix A:")
print(A_2x2)
print()

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = compute_eigensystem(A_2x2)

print("Eigenvalues:")
print(eigenvalues)
print()

print("Eigenvectors:")
print(eigenvectors)
print()

# Verify: Av = λv
print("Verification (Av = λv):")
for i, eigenvalue in enumerate(eigenvalues):
    v = eigenvectors[:, i]
    Av = A_2x2 @ v
    lambda_v = eigenvalue * v
    print()
    print(f"Eigenvalue {i+1}: λ = {eigenvalue:.4f}")
    print(f"Av = {Av}")
    print(f"λv = {lambda_v}")
    print(f"Difference: {np.linalg.norm(Av - lambda_v):.2e}")


Matrix A:
[[ 4. -2.]
 [ 1.  1.]]

Eigenvalues:
[3. 2.]

Eigenvectors:
[[0.89442719+0.j 0.70710678+0.j]
 [0.4472136 +0.j 0.70710678+0.j]]

Verification (Av = λv):

Eigenvalue 1: λ = 3.0000
Av = [2.68328157+0.j 1.34164079+0.j]
λv = [2.68328157+0.j 1.34164079+0.j]
Difference: 2.22e-16

Eigenvalue 2: λ = 2.0000
Av = [1.41421356+0.j 1.41421356+0.j]
λv = [1.41421356+0.j 1.41421356+0.j]
Difference: 3.14e-16


## Example 2: 3x3 Symmetric Matrix

In [7]:
# Define a 3x3 symmetric matrix
A_3x3 = np.array([
    [2, 1, 0],
    [1, 2, 1],
    [0, 1, 2]
], dtype=float)

print("Matrix A:")
print(A_3x3)
print()

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = compute_eigensystem(A_3x3)

print("Eigenvalues:")
print(eigenvalues)
print()

print("Eigenvectors (as columns):")
print(eigenvectors)
print()

# Verify for first eigenvalue
print("Verification (Av = λv) for first eigenvalue:")
v = eigenvectors[:, 0]
Av = A_3x3 @ v
lambda_v = eigenvalues[0] * v
print(f"Av = {Av}")
print(f"λv = {lambda_v}")
print(f"Error: {np.linalg.norm(Av - lambda_v):.2e}")

Matrix A:
[[2. 1. 0.]
 [1. 2. 1.]
 [0. 1. 2.]]

Eigenvalues:
[3.41421356 2.         0.58578644]

Eigenvectors (as columns):
[[ 5.00000000e-01+0.j -7.07106781e-01+0.j  5.00000000e-01+0.j]
 [ 7.07106781e-01+0.j -1.57009246e-15+0.j -7.07106781e-01+0.j]
 [ 5.00000000e-01+0.j  7.07106781e-01+0.j  5.00000000e-01+0.j]]

Verification (Av = λv) for first eigenvalue:
Av = [1.70710678+0.j 2.41421356+0.j 1.70710678+0.j]
λv = [1.70710678+0.j 2.41421356+0.j 1.70710678+0.j]
Error: 1.89e-14


## Comparison with NumPy's Built-in Function

In [8]:
# Compare with numpy's linalg.eig
print("Using numpy.linalg.eig:")
np_eigenvalues, np_eigenvectors = np.linalg.eig(A_3x3)

print("NumPy Eigenvalues:")
print(np_eigenvalues)
print()

print("NumPy Eigenvectors (as columns):")
print(np_eigenvectors)
print()

print()
print("Our Implementation Eigenvalues:")
print(eigenvalues)
print()

print("Difference in eigenvalues:")
print(np.linalg.norm(np_eigenvalues - eigenvalues))


Using numpy.linalg.eig:
NumPy Eigenvalues:
[3.41421356 2.         0.58578644]

NumPy Eigenvectors (as columns):
[[-5.00000000e-01  7.07106781e-01  5.00000000e-01]
 [-7.07106781e-01  4.05405432e-16 -7.07106781e-01]
 [-5.00000000e-01 -7.07106781e-01  5.00000000e-01]]


Our Implementation Eigenvalues:
[3.41421356 2.         0.58578644]

Difference in eigenvalues:
6.2685830808342686e-15
